# Episode 20: Web Browsing

An agent's knowledge is frozen at training time. To answer *"what's on this page right now?"* it needs to reach the live web.

**In this episode you'll build:** an agent with a web-fetch tool that retrieves and summarizes a page.

## Web access is just a tool

There's nothing special about web access — it's a **tool**, exactly like the ones in Episode 3. You write a function that fetches a URL; the agent decides when to call it.

We build that tool with `httpx`, a standard async-friendly HTTP client (it's in the workshop's `requirements.txt`).

## Setup

In [ ]:
from dotenv import load_dotenv

load_dotenv()

import httpx

from autogen.beta import Agent
from autogen.beta.config import OpenAIConfig

config = OpenAIConfig(model="gpt-4.1-mini", temperature=0)

## Step 1: A web-fetch tool

The tool is a plain function: take a URL, fetch it, return the text. We cap the length so a huge page doesn't blow the context window — real tools trim and clean their output.

In [ ]:
def fetch_url(url: str) -> str:
    """Fetch the text content of a web page given its URL."""
    response = httpx.get(url, timeout=15.0, follow_redirects=True)
    response.raise_for_status()
    return response.text[:3000]

## Step 2: An agent that browses

Give the agent the tool and ask a question that requires fetching a page. The agent calls `fetch_url`, reads the result, and answers.

In [1]:
web_agent = Agent(
    "web_agent",
    prompt="Use fetch_url to retrieve a page, then answer the user's question about it.",
    config=config,
    tools=[fetch_url],
)

reply = await web_agent.ask(
    "Fetch https://example.com and tell me in one sentence what the page is for."
)
print("ANSWER:", reply.body)

ANSWER: The page "https://example.com" is for use in documentation examples without needing permission and is not intended for operational use.


## What just happened

The agent had no idea what `example.com` contained. It called `fetch_url`, received the raw HTML, and summarized it. The same pattern reaches any HTTP API — a weather endpoint, a JSON feed, your own internal service.

The model decided *when* to fetch and *what* URL to pass, from the docstring and the question — the routing lesson from Episode 3, applied to the open web.

## Additional content

**Provider-native web tools.** AG2 Beta also ships `WebSearchTool` and `WebFetchTool` in `autogen.beta.tools`. These are *provider-native* — they run inside an Anthropic model's own server-side web tooling, so they require an Anthropic-backed `Agent`. With an OpenAI config you write the tool yourself, as we did here.

**Search vs fetch.** `fetch_url` retrieves a *known* URL. To *find* pages you also need search — wrap a search API (Tavily, Brave, DuckDuckGo) in the same way: a function that takes a query and returns results.

**Clean your output.** Returning raw HTML works but wastes tokens. Production web tools strip tags, extract the main content, and often summarize before returning — keeping the agent's context lean (see Episodes 7–8).

## What's next

Your agents can reach knowledge and the web. Episodes 21–22 give them a **face** — a web UI users can actually talk to.